# Explore bronze Data

In [2]:
from processing.spark.utils.spark_session import create_spark_session

In [3]:
spark = create_spark_session("explore-bronze")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 18:24:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Movies

In [46]:
movies = spark.read.format("delta").load("s3a://bronze/batch/movies")

In [47]:
movies.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [48]:
movies.limit(10).show(truncate = False)

+--------+----------------------------------+-------------------------------------------+------------------------+--------+
|movie_id|title                             |genres                                     |_ingestion_timestamp    |_source |
+--------+----------------------------------+-------------------------------------------+------------------------+--------+
|1       |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|2026-03-31 14:40:37.8339|postgres|
|2       |Jumanji (1995)                    |Adventure|Children|Fantasy                 |2026-03-31 14:40:37.8339|postgres|
|3       |Grumpier Old Men (1995)           |Comedy|Romance                             |2026-03-31 14:40:37.8339|postgres|
|4       |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |2026-03-31 14:40:37.8339|postgres|
|5       |Father of the Bride Part II (1995)|Comedy                                     |2026-03-31 14:40:37.8339|postgres|
|6      

In [49]:
movies.describe().show()

+-------+------------------+---------------+------------------+--------+
|summary|          movie_id|          title|            genres| _source|
+-------+------------------+---------------+------------------+--------+
|  count|            105071|         105071|            104629|  105071|
|   mean|168749.16193811802|           40.0|              null|    null|
| stddev| 81494.36957094916|           null|              null|    null|
|    min|                 1|         (1993)|(no genres listed)|postgres|
|    max|            300373|줄탁동시 (2012)|           Western|postgres|
+-------+------------------+---------------+------------------+--------+



26/03/31 14:48:05 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-13956540-4e8e-4743-940f-1dbb3765c047. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-13956540-4e8e-4743-940f-1dbb3765c047
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:177)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:113)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:94)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1231)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:65)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4$adapted(ShutdownHookManager.scala:62)
	at scala.collection.IndexedSeqOptimized.foreach(IndexedSeqOptimized.scala:36)
	at scala.collection.IndexedSeqOptimized.foreach$(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofRef.foreach(Array

**Colunas**
movieId

- Identificador único do filme.

title

- Título do filme.

genres

- Gêneros do filme. Quando há mais de um, eles aparecem separados pelo caractere “|”.


**Separar genres em tabela**

Tabela principal (silver.cleaned.movies)

```
movie_id
title
release_year
ingestion_timestamp
source
```

Tabela de relacionamento (silver.cleaned.movie_genres)
```
movie_id
genre
```

**Padronização de colunas**
```
movie_id -> movie_id
title -> title
genres -> genres_raw
```

**Separar título e ano**


```
Toy Story (1995)

title = Toy Story
release_year = 1995
```


**Tratar valores nulos**

```title NULL``` -> descartar (dado inválido)

```genres NULL``` -> preencher como ```unknown```

**Tipagem correta**

<table align="left">
  <tr>
    <th>coluna</th>
    <th>tipo</th>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>int</td>
  </tr>
  <tr>
    <td>title</td>
    <td>string</td>
  </tr>
  <tr>
    <td>release_year</td>
    <td>int</td>
  </tr>
  <tr>
    <td>genre</td>
    <td>string</td>
  </tr>
</table>

**Particionamento**

release_year

## Belief Data

In [12]:
belief = spark.read.parquet("s3a://bronze/batch/belief_data")

In [13]:
belief.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- is_seen: short (nullable = true)
 |-- watch_date: string (nullable = true)
 |-- user_elicit_rating: double (nullable = true)
 |-- user_predict_rating: double (nullable = true)
 |-- user_certainty: double (nullable = true)
 |-- tstamp: timestamp (nullable = true)
 |-- movie_idx: integer (nullable = true)
 |-- source: integer (nullable = true)
 |-- system_predict_rating: double (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [14]:
belief.limit(10).show(truncate = False)

+-------+--------+-------+----------+------------------+-------------------+--------------+-------------------+---------+------+---------------------+--------------------------+--------+
|user_id|movie_id|is_seen|watch_date|user_elicit_rating|user_predict_rating|user_certainty|tstamp             |movie_idx|source|system_predict_rating|_ingestion_timestamp      |_source |
+-------+--------+-------+----------+------------------+-------------------+--------------+-------------------+---------+------+---------------------+--------------------------+--------+
|53982  |1       |-1     |          |-1.0              |-1.0               |-1.0          |2023-05-01 18:59:04|2        |2     |4.44785083170358     |2026-03-31 14:40:52.370411|postgres|
|56737  |1       |-1     |          |-1.0              |-1.0               |-1.0          |2023-10-08 13:52:36|7        |1     |3.57615421715505     |2026-03-31 14:40:52.370411|postgres|
|57704  |1       |-1     |          |-1.0              |-1.0     

In [15]:
belief.describe().show()

+-------+-----------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+---------------------+--------+
|summary|          user_id|          movie_id|            is_seen|         watch_date| user_elicit_rating|user_predict_rating|     user_certainty|         movie_idx|            source|system_predict_rating| _source|
+-------+-----------------+------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+---------------------+--------+
|  count|          3004084|           3004084|            3004084|            3004084|            3004084|            3004084|            3004084|           3004084|           3004084|              3004084| 3004084|
|   mean|339709.6425702477|118793.36241563152|-0.9848233271772694|               null|-0.9765136394321863|-0.9614321703387788|-0.9585940

**Colunas**

userId

- Identificador do usuário.

movieId

- Identificador do filme.

isSeen

- Indica se o usuário já viu o filme.
```
-1 = não respondeu
0 = não viu
1 = já viu
```

watchDate

- Data aproximada em que o usuário assistiu ao filme. Só possui valor quando isSeen = 1.

userElicitRating

- Nota real informada pelo usuário para o filme quando ele já assistiu.

```Escala de 0.5 a 5.0.```

userPredictRating

- Nota que o usuário acredita que daria ao filme caso ainda não tenha assistido.

```Escala de 0.5 a 5.0.```

userCertainty

- Nível de confiança do usuário na previsão da nota.

```Escala de 1 a 5.```

tstamp

- Timestamp do momento em que a informação foi registrada.

month_idx

- Índice que representa o mês do experimento em que o filme foi incluído no processo de elicitação.

source

- Indica de qual grupo do processo de seleção o filme foi escolhido.

systemPredictRating

- Nota prevista pelo sistema de recomendação para aquele usuário e filme

**Padronização de colunas**

<table align="left">
  <tr>
    <th>original</th>
    <th>novo</th>
  </tr>
  <tr>
    <td>user_id</td>
    <td>user_id</td>
  </tr>
  <tr>
    <td>movie_id</td>
    <td>movie_id</td>
  </tr>
  <tr>
    <td>is_seen</td>
    <td>is_seen</td>
  </tr>
  <tr>
    <td>watch_date</td>
    <td>watch_date</td>
  </tr>
  <tr>
    <td>user_elicit_rating</td>
    <td>user_rating</td>
  </tr>
  <tr>
    <td>user_predict_rating</td>
    <td>predicted_rating</td>
  </tr>
  <tr>
    <td>user_certainty</td>
    <td>user_certainty</td>
  </tr>  
  <tr>
    <td>system_predict_rating</td>
    <td>system_rating</td>
  </tr>
  <tr>
    <td>tstamp</td>
    <td>event_timestamp</td>
  </tr>
  <tr>
    <td>movie_idx</td>
    <td>movie_idx</td>
  </tr>
  <tr>
    <td>source</td>
    <td>source</td>
  </tr>
  <tr>
    <td>_ingestion_timestamp</td>
    <td>ingestion_timestamp</td>
  </tr>
  <tr>
    <td>_source</td>
    <td>source_system</td>
  </tr>
</table>

**Tratamento de dados inválidos**
<table align="left"> 
    <tr> 
        <th>coluna</th> 
        <th>regra</th> 
    </tr> <tr> 
        <td>is_seen</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>user_rating</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>predicted_rating</td> 
        <td>-1 = NULL</td> 
    </tr> <tr> 
        <td>user_certainty</td> 
        <td>-1 = NULL</td> 
    </tr> </table>

**Tratamento de datas**

"" = NULL

cast para date

**Remoção de dados inválidos**

descartados:

- user_id NULL
- movie_id NULL

**Deduplicação**

(user_id, movie_id, event_timestamp)

manter o mais recente via _ingestion_timestamp

**Tipagem final**
<table align="left"> <tr> <th>coluna</th> <th>tipo</th> </tr> <tr> <td>user_id</td> <td>int</td> </tr> <tr> <td>movie_id</td> <td>int</td> </tr> <tr> <td>is_seen</td> <td>int</td> </tr> <tr> <td>watch_date</td> <td>date</td> </tr> <tr> <td>user_rating</td> <td>double</td> </tr> <tr> <td>predicted_rating</td> <td>double</td> </tr> <tr> <td>system_rating</td> <td>double</td> </tr> <tr> <td>user_certainty</td> <td>double</td> </tr> <tr> <td>event_timestamp</td> <td>timestamp</td> </tr> <tr> <td>movie_idx</td> <td>int</td> </tr> <tr> <td>source</td> <td>int</td> </tr> <tr> <td>ingestion_timestamp</td> <td>timestamp</td> </tr> <tr> <td>processed_timestamp</td> <td>timestamp</td> </tr> </table>

## Movie Elicitation

In [16]:
elicitation = spark.read.parquet("s3a://bronze/batch/movie_elicitation_set")

In [17]:
elicitation.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- month_idx: integer (nullable = true)
 |-- source: integer (nullable = true)
 |-- tstamp: timestamp (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)



In [23]:
elicitation.limit(10).show(truncate = False)

+--------+---------+------+-------------------+--------------------------+--------+
|movie_id|month_idx|source|tstamp             |_ingestion_timestamp      |_source |
+--------+---------+------+-------------------+--------------------------+--------+
|1       |0        |1     |2023-02-27 19:30:03|2026-03-31 12:43:44.493826|postgres|
|1       |1        |1     |2023-04-01 00:01:47|2026-03-31 12:43:44.493826|postgres|
|1       |2        |1     |2023-05-01 00:02:01|2026-03-31 12:43:44.493826|postgres|
|1       |3        |1     |2023-06-01 00:01:40|2026-03-31 12:43:44.493826|postgres|
|1       |4        |1     |2023-07-01 00:01:56|2026-03-31 12:43:44.493826|postgres|
|1       |5        |1     |2023-08-01 00:01:40|2026-03-31 12:43:44.493826|postgres|
|2       |0        |1     |2023-02-27 19:30:03|2026-03-31 12:43:44.493826|postgres|
|2       |1        |1     |2023-04-01 00:01:47|2026-03-31 12:43:44.493826|postgres|
|2       |2        |1     |2023-05-01 00:02:01|2026-03-31 12:43:44.493826|po

## Ratings

In [19]:
ratings = spark.read.parquet("s3a://bronze/streaming/ratings_for_additional_users")

In [20]:
ratings.printSchema()

root
 |-- userId: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- tstamp: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)
 |-- _topic: string (nullable = true)



In [22]:
ratings.limit(10).show(truncate = False)

+------+-------+------+-------------------+--------------------------+-------+--------------------------------+
|userId|movieId|rating|tstamp             |_ingestion_timestamp      |_source|_topic                          |
+------+-------+------+-------------------+--------------------------+-------+--------------------------------+
|328221|187505 |5     |2018-12-05 23:10:11|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187541 |5     |2018-11-06 21:39:15|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187577 |5     |2022-02-22 14:20:19|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187593 |4     |2018-10-12 08:20:55|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187601 |5     |2018-12-05 23:33:37|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional_users|
|328221|187717 |4     |2020-10-26 14:35:06|2026-03-30 18:19:59.541793|kafka  |csv-ratings_for_additional

## Recommendation History

In [25]:
history = spark.read.parquet("s3a://bronze/streaming/user_recommendation_history")

In [26]:
history.printSchema()

root
 |-- userId: string (nullable = true)
 |-- tstamp: string (nullable = true)
 |-- movieId: string (nullable = true)
 |-- predictedRating: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source: string (nullable = true)
 |-- _topic: string (nullable = true)



In [28]:
history.limit(10).show(truncate = False)

+------+----------+-------+----------------+--------------------------+-------+-------------------------------+
|userId|tstamp    |movieId|predictedRating |_ingestion_timestamp      |_source|_topic                         |
+------+----------+-------+----------------+--------------------------+-------+-------------------------------+
|377084|1677651051|296    |4.62638615403935|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|55820  |4.34364173570518|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|924    |4.30338526694785|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1732   |4.2554163884785 |2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1258   |4.24126920909493|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_history|
|377084|1677651051|1201   |4.21599746536257|2026-03-30 18:26:04.051121|kafka  |csv-user_recommendation_h